## Setup — enabling the Rust extension

`monod_core` is a compiled Rust extension built with [maturin](https://www.maturin.rs/).
If it is not installed the notebook falls back to pure Python automatically, but all
benchmarks will show **1× speedup** (Python vs Python).

### Quick install (3 steps)

**Step 1 — install the Rust toolchain** (skip if `rustc --version` already works):

```bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
source ~/.cargo/env          # or restart your terminal
```

**Step 2 — install maturin** inside the Python environment you use for this notebook:

```bash
pip install maturin
# or, if you manage packages with conda:
# conda install -c conda-forge maturin
```

**Step 3 — build and install `monod_core`** from the repo root:

```bash
cd monod_core
maturin develop --release   # builds and installs into the active Python env
```

> **Apple Silicon (M-series) users** running an x86-64 Python under Rosetta:  
> replace `maturin develop` with `maturin build --release -i $(which python3)` and
> `pip install target/wheels/monod_core-*.whl`.

The cell below checks your installation status.

In [ ]:
import importlib, subprocess, sys, os, pathlib

def _try_run(*args):
    """Run a command; return stdout string or None on failure."""
    # Search ~/.cargo/bin and the active Python env's bin (covers conda envs)
    env = os.environ.copy()
    extra = [str(pathlib.Path.home() / ".cargo" / "bin"),
             str(pathlib.Path(sys.prefix) / "bin")]
    env["PATH"] = os.pathsep.join(extra) + os.pathsep + env.get("PATH", "")
    try:
        return subprocess.check_output(list(args), stderr=subprocess.DEVNULL,
                                       text=True, env=env).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


def _check_rust_setup():
    # ── 1. Is monod_core importable? ─────────────────────────────────────────
    spec = importlib.util.find_spec("monod_core")
    if spec is None:
        print("monod_core NOT found — Rust acceleration is disabled.")
        print()
        print("To enable it, run the following commands from the repo root:")
        print()
        print("  # Step 1: install Rust (skip if `rustc --version` already works)")
        print("  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh")
        print("  source ~/.cargo/env")
        print()
        print("  # Step 2: install maturin into this Python environment")
        print("  pip install maturin")
        print()
        print("  # Step 3: build and install the extension")
        print("  cd monod_core")
        print("  maturin develop --release")
        print()
        print("Then restart the kernel and re-run this notebook.")
        return False

    import monod_core as _mc
    print(f"monod_core  : {_mc.__file__}")

    rustc = _try_run("rustc", "--version")
    print(f"rustc       : {rustc}" if rustc else
          "rustc       : not found — add ~/.cargo/bin to PATH to rebuild")

    maturin = _try_run("maturin", "--version")
    print(f"maturin     : {maturin}" if maturin else
          "maturin     : not found — install with: pip install maturin")

    print()
    print("Rust extension is ready.  All benchmarks will use the Rust fast-path.")
    return True


rust_ready = _check_rust_setup()

# Rust vs Python: Unique Histogram Extraction

This notebook benchmarks `make_histograms_unique` — the core data-extraction step in
the monod pipeline — comparing the Rust/rayon implementation against the pure-Python/numpy
baseline.

**What the function does:**  
For each gene (column), across *n_layers* count matrices each of shape *(n_cells × n_genes)*,
find every unique observed microstate (e.g. `(spliced_count, unspliced_count)`) and its
frequency. This is the `hist_type='unique'` path in `extract_data`.

**Rust optimisations applied:**
1. `rayon::par_iter` over genes — parallel across all CPU cores, GIL released
2. Per-gene contiguous column copy — eliminates stride-*n_genes* cache misses
3. Counting sort for small count ranges — O(n_cells + max_u×max_s) instead of O(n log n)
4. Thread-local scratch buffers — zero per-gene allocations in the hot path

In [ ]:
import sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import anndata as ad
from scipy.sparse import issparse

sys.path.insert(0, '../src')
import monod_core as _mc

warnings.filterwarnings('ignore')
print(f'monod_core loaded: {_mc.__file__}')

## Helper functions

In [ ]:
def py_make_histograms_unique(layers):
    """Pure-Python reference: serial loop + np.unique per gene."""
    n_cells = layers[0].shape[0]
    coords_all, freqs_all = [], []
    for g in range(layers[0].shape[1]):
        stacked = np.column_stack([layer[:, g] for layer in layers])
        unique, counts = np.unique(stacked, axis=0, return_counts=True)
        coords_all.append(unique)
        freqs_all.append(counts / n_cells)
    return coords_all, freqs_all


def rust_make_histograms_unique(layers):
    """Rust/rayon path via monod_core extension."""
    int64_layers = [np.ascontiguousarray(l, dtype=np.int64) for l in layers]
    coords_raw, freqs_raw = _mc.make_histograms_unique(int64_layers)
    return [np.array(c, dtype=np.int64) for c in coords_raw], \
           [np.array(f)                 for f in freqs_raw]


def bench(fn, layers, n_reps=5):
    fn(layers)           # warm-up
    t0 = time.perf_counter()
    for _ in range(n_reps):
        fn(layers)
    return (time.perf_counter() - t0) / n_reps * 1e3  # ms

## Correctness check

In [ ]:
rng = np.random.default_rng(42)
l1 = rng.integers(0, 10, (200, 50))
l2 = rng.integers(0, 10, (200, 50))

py_c, py_f = py_make_histograms_unique([l1, l2])
ru_c, ru_f = rust_make_histograms_unique([l1, l2])

for g in range(50):
    assert np.array_equal(ru_c[g], py_c[g]), f'gene {g}: coord mismatch'
    assert np.allclose(ru_f[g],   py_f[g]),  f'gene {g}: freq mismatch'

print('Rust and Python outputs are identical for all 50 genes')

## Benchmark 1 — scaling with number of genes

Fixed: 5 000 cells, max count = 20 (typical RNA-seq range — triggers counting-sort path).

In [ ]:
rng = np.random.default_rng(0)
N_CELLS, MAX_COUNT = 5_000, 20
gene_counts = [10, 25, 50, 100, 200, 400]

py_times, ru_times = [], []
for n_genes in gene_counts:
    l1 = rng.integers(0, MAX_COUNT, (N_CELLS, n_genes))
    l2 = rng.integers(0, MAX_COUNT, (N_CELLS, n_genes))
    py_times.append(bench(py_make_histograms_unique, [l1, l2]))
    ru_times.append(bench(rust_make_histograms_unique, [l1, l2]))
    speedup = py_times[-1] / ru_times[-1]
    print(f'n_genes={n_genes:4d}  '
          f'Python={py_times[-1]:7.1f} ms  '
          f'Rust={ru_times[-1]:6.1f} ms  '
          f'speedup={speedup:.1f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(gene_counts, py_times, 'o-', color='steelblue', label='Python (numpy, serial)')
ax.plot(gene_counts, ru_times, 's-', color='firebrick', label='Rust (rayon, parallel)')
ax.set_xlabel('Number of genes')
ax.set_ylabel('Wall time (ms)')
ax.set_title(f'{N_CELLS:,} cells, max count = {MAX_COUNT}')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
speedups = [p / r for p, r in zip(py_times, ru_times)]
bar_w = (gene_counts[-1] - gene_counts[0]) / len(gene_counts) * 0.6
ax.bar(gene_counts, speedups, color='seagreen', alpha=0.8, width=bar_w)
ax.axhline(1, color='grey', lw=0.8, ls='--')
ax.set_xlabel('Number of genes')
ax.set_ylabel('Speedup')
ax.set_title('Speedup: Python / Rust')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0fx'))
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Unique-histogram extraction: Rust vs Python', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## Benchmark 2 — scaling with number of cells

Fixed: 100 genes, max count = 20.

In [ ]:
rng = np.random.default_rng(1)
N_GENES, MAX_COUNT = 100, 20
cell_counts = [500, 1_000, 2_000, 5_000, 10_000, 20_000]

py_times2, ru_times2 = [], []
for n_cells in cell_counts:
    l1 = rng.integers(0, MAX_COUNT, (n_cells, N_GENES))
    l2 = rng.integers(0, MAX_COUNT, (n_cells, N_GENES))
    py_times2.append(bench(py_make_histograms_unique, [l1, l2], n_reps=3))
    ru_times2.append(bench(rust_make_histograms_unique, [l1, l2], n_reps=3))
    speedup = py_times2[-1] / ru_times2[-1]
    print(f'n_cells={n_cells:6d}  '
          f'Python={py_times2[-1]:7.1f} ms  '
          f'Rust={ru_times2[-1]:6.1f} ms  '
          f'speedup={speedup:.1f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(cell_counts, py_times2, 'o-', color='steelblue', label='Python (numpy, serial)')
ax.plot(cell_counts, ru_times2, 's-', color='firebrick', label='Rust (rayon, parallel)')
ax.set_xlabel('Number of cells')
ax.set_ylabel('Wall time (ms)')
ax.set_title(f'{N_GENES} genes, max count = {MAX_COUNT}')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
speedups2 = [p / r for p, r in zip(py_times2, ru_times2)]
ax.plot(cell_counts, speedups2, 'D-', color='darkorange')
ax.axhline(1, color='grey', lw=0.8, ls='--')
ax.set_xlabel('Number of cells')
ax.set_ylabel('Speedup')
ax.set_title('Speedup: Python / Rust')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0fx'))
ax.grid(True, alpha=0.3)

fig.suptitle('Scaling with cell count', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## Benchmark 3 — effect of count magnitude

The Rust implementation uses **counting sort** when `(max_l0+1)×(max_l1+1) ≤ 16 384`
and falls back to comparison sort for larger ranges.
For 2 layers the boundary is at max_count ≈ 128 per layer (√16384).

In [ ]:
rng = np.random.default_rng(2)
N_CELLS3, N_GENES3 = 5_000, 100
# 128*128 = 16384 = DENSE_THRESHOLD, so 127 is the last value in the counting path
max_counts3 = [5, 10, 20, 50, 100, 127, 128, 200, 500, 1000]

py_times3, ru_times3 = [], []
for mc_val in max_counts3:
    l1 = rng.integers(0, mc_val, (N_CELLS3, N_GENES3))
    l2 = rng.integers(0, mc_val, (N_CELLS3, N_GENES3))
    py_times3.append(bench(py_make_histograms_unique, [l1, l2]))
    ru_times3.append(bench(rust_make_histograms_unique, [l1, l2]))
    path = 'counting' if mc_val * mc_val <= 16384 else 'sort   '
    speedup = py_times3[-1] / ru_times3[-1]
    print(f'max={mc_val:5d}  [{path}]  '
          f'Python={py_times3[-1]:7.1f} ms  '
          f'Rust={ru_times3[-1]:6.1f} ms  '
          f'speedup={speedup:.1f}x')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(max_counts3, py_times3, 'o-', color='steelblue', label='Python (numpy, serial)')
ax.plot(max_counts3, ru_times3, 's-', color='firebrick', label='Rust (rayon, parallel)')

ax.axvline(127, color='grey', ls=':', lw=1.2)
ylim = ax.get_ylim()
ax.text(135, ylim[0] + (ylim[1] - ylim[0]) * 0.75,
        'counting sort\n→ comparison sort', color='grey', fontsize=8)

ax.set_xlabel('Max count value per layer')
ax.set_ylabel('Wall time (ms)')
ax.set_title(f'Effect of count magnitude — {N_CELLS3:,} cells × {N_GENES3} genes')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Benchmark 4 — real data (gaba_example.h5ad)

728 cells × up to 200 selected genes from the spliced/unspliced layers.

In [ ]:
adata_full = ad.read_h5ad('example_h5ad/gaba_example.h5ad')
spliced   = adata_full.layers['spliced']
unspliced = adata_full.layers['unspliced']
s_dense = spliced.toarray()   if issparse(spliced)   else np.asarray(spliced)
u_dense = unspliced.toarray() if issparse(unspliced) else np.asarray(unspliced)

print(f'Dataset: {adata_full.n_obs} cells × {adata_full.n_vars} genes')
print(f'Spliced   — dtype: {s_dense.dtype}, max: {int(s_dense.max())}')
print(f'Unspliced — dtype: {u_dense.dtype}, max: {int(u_dense.max())}')

In [ ]:
rng = np.random.default_rng(3)
expressed = np.where((s_dense > 0).sum(0) >= 10)[0]
gene_subsets4 = [10, 25, 50, 100, 200]

py_times4, ru_times4 = [], []
for n_g in gene_subsets4:
    idx = rng.choice(expressed, size=min(n_g, len(expressed)), replace=False)
    s_sub = s_dense[:, idx].astype(np.int64)
    u_sub = u_dense[:, idx].astype(np.int64)
    py_times4.append(bench(py_make_histograms_unique, [s_sub, u_sub], n_reps=3))
    ru_times4.append(bench(rust_make_histograms_unique, [s_sub, u_sub], n_reps=3))
    speedup = py_times4[-1] / ru_times4[-1]
    print(f'n_genes={n_g:4d}  '
          f'Python={py_times4[-1]:7.1f} ms  '
          f'Rust={ru_times4[-1]:6.1f} ms  '
          f'speedup={speedup:.1f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(gene_subsets4, py_times4, 'o-', color='steelblue', label='Python (numpy, serial)')
ax.plot(gene_subsets4, ru_times4, 's-', color='firebrick', label='Rust (rayon, parallel)')
ax.set_xlabel('Number of genes')
ax.set_ylabel('Wall time (ms)')
ax.set_title(f'gaba_example.h5ad ({adata_full.n_obs} cells)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
speedups4 = [p / r for p, r in zip(py_times4, ru_times4)]
ax.bar(gene_subsets4, speedups4, color='mediumpurple', alpha=0.85, width=12)
ax.axhline(1, color='grey', lw=0.8, ls='--')
ax.set_xlabel('Number of genes')
ax.set_ylabel('Speedup')
ax.set_title('Speedup: Python / Rust')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0fx'))
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Real scRNA-seq data — gaba_example.h5ad', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## Output format

A look at what the function returns for one real gene.

In [ ]:
import pandas as pd

gene_idx  = expressed[0]
gene_name = adata_full.var_names[gene_idx]
s_gene = s_dense[:, gene_idx:gene_idx+1].astype(np.int64)
u_gene = u_dense[:, gene_idx:gene_idx+1].astype(np.int64)

coords, freqs = rust_make_histograms_unique([s_gene, u_gene])

df = pd.DataFrame(coords[0], columns=['spliced', 'unspliced'])
df['frequency'] = freqs[0]
df = df.sort_values('frequency', ascending=False).head(10).reset_index(drop=True)

print(f'Gene: {gene_name}  —  '
      f'{len(coords[0])} unique microstates across {adata_full.n_obs} cells')
print()
print(df.to_string(index=False))

## Summary table

In [ ]:
header = f'{"Scenario":<50} {"Python (ms)":>12} {"Rust (ms)":>10} {"Speedup":>9}'
print(header)
print('-' * len(header))

rows = [
    ('Synthetic  5k cells × 200 genes, max=20  [counting]',
     py_times[gene_counts.index(200)], ru_times[gene_counts.index(200)]),
    ('Synthetic 20k cells × 100 genes, max=20  [counting]',
     py_times2[-1], ru_times2[-1]),
    ('Synthetic  5k cells × 100 genes, max=1000 [sort]',
     py_times3[-1], ru_times3[-1]),
    ('Real data (gaba)  728 cells × 200 genes',
     py_times4[-1], ru_times4[-1]),
]
for label, py_t, ru_t in rows:
    print(f'{label:<50} {py_t:>12.1f} {ru_t:>10.1f} {py_t/ru_t:>8.1f}x')

## Benchmark 5 — model evaluation (`eval_model_pss_2d`)

Each call to `eval_model_pss_2d` computes the steady-state probability distribution for one
set of parameters on a 2-D grid. It is called hundreds of times per gene during L-BFGS-B
optimization. Rust replaces numpy quadrature + `scipy.fft.irfftn` with rayon-parallel
Gauss-Legendre integration and a parallel IFFT.

This benchmark sweeps over grid size (the dominant cost factor).

In [ ]:
import logging
import sys
sys.path.insert(0, 'src/monod')

import extract_data as _ed_mod
import cme_toolbox as _ct_mod
from cme_toolbox import CMEModel
from extract_data import extract_data
from inference import InferenceParameters, searchdata_from_adata

logging.getLogger().setLevel(logging.WARNING)

# ── Shared model ─────────────────────────────────────────────────────────────
BENCH_MODEL  = CMEModel("Bursty", "Poisson")

# ── PSS microbenchmark constants ─────────────────────────────────────────────
P_LOG        = [0.0, 0.0, 0.0]          # b=1, beta=1, gamma=1 (log10 scale)
SAMP_LOG     = [1.0, 1.0]               # Poisson capture rate 10 (log10=1)
FIXED_QUAD_T = float(BENCH_MODEL.fixed_quad_T)
QUAD_ORDER   = int(BENCH_MODEL.quad_order)

# ── Full-pipeline benchmark constants ────────────────────────────────────────
BENCH_GENES  = ["Eif5b", "Xrcc5", "Klhl12", "Rgs7", "Thsd7b"]
adata_ref5   = ad.read_h5ad('example_h5ad/gaba_results.h5ad')
BENCH_LENGTHS = adata_ref5.var.loc[BENCH_GENES, "log_lengths"].values

print(f"Model        : {BENCH_MODEL.bio_model} + {BENCH_MODEL.seq_model}")
print(f"fixed_quad_T : {FIXED_QUAD_T},  quad_order: {QUAD_ORDER}")
print(f"Pipeline genes: {BENCH_GENES}")

In [ ]:
import monod_core as _mc_direct

def bench_pss(sz, n_reps=10):
    """Time one eval_model_pss_2d call for an sz×sz grid in Python and Rust."""
    p_arr    = np.array(P_LOG)
    lims_arr = np.array([sz, sz], dtype=np.intp)
    lims_int = [sz, sz]
    samp_arr = np.array(SAMP_LOG)

    # Python path: bypass Rust
    _ct_mod._HAS_RUST = False
    BENCH_MODEL.eval_model_pss(p_arr, lims_arr, samp=samp_arr)  # warm-up
    t0 = time.perf_counter()
    for _ in range(n_reps):
        BENCH_MODEL.eval_model_pss(p_arr, lims_arr, samp=samp_arr)
    t_py = (time.perf_counter() - t0) / n_reps * 1e3
    _ct_mod._HAS_RUST = True

    # Rust path: direct monod_core call
    _mc_direct.eval_model_pss_2d(
        "Bursty", P_LOG, lims_int, FIXED_QUAD_T, QUAD_ORDER, SAMP_LOG)  # warm-up
    t0 = time.perf_counter()
    for _ in range(n_reps):
        _mc_direct.eval_model_pss_2d(
            "Bursty", P_LOG, lims_int, FIXED_QUAD_T, QUAD_ORDER, SAMP_LOG)
    t_ru = (time.perf_counter() - t0) / n_reps * 1e3

    return t_py, t_ru


grid_sizes5 = [20, 50, 100, 200, 500, 1000]
py_pss, ru_pss = [], []
for sz in grid_sizes5:
    tp, tr = bench_pss(sz)
    py_pss.append(tp)
    ru_pss.append(tr)
    print(f"limits=({sz:4d},{sz:4d})  Python={tp:8.2f} ms  Rust={tr:7.3f} ms  "
          f"speedup={tp/tr:.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(grid_sizes5, py_pss, 'o-', color='steelblue', label='Python (scipy quadrature + irfftn)')
ax.plot(grid_sizes5, ru_pss, 's-', color='firebrick', label='Rust (rayon GL quadrature + IFFT)')
ax.set_xlabel('Grid size (N×N)')
ax.set_ylabel('Wall time per call (ms)')
ax.set_title('eval_model_pss_2d — Bursty+Poisson')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
speedups_pss = [p / r for p, r in zip(py_pss, ru_pss)]
ax.plot(grid_sizes5, speedups_pss, 'D-', color='darkorange')
ax.axhline(1, color='grey', lw=0.8, ls='--')
ax.set_xlabel('Grid size (N×N)')
ax.set_ylabel('Speedup (Python / Rust)')
ax.set_title('Speedup — eval_model_pss_2d')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0fx'))
ax.grid(True, alpha=0.3)

fig.suptitle('Model evaluation speedup (per PSS call)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
INFER_PARAMS = dict(
    use_lengths=True,
    gradient_params={"max_iterations": 15, "init_pattern": "moments", "num_restarts": 1},
    gridsize=[3, 3],
    save=False,
)


def run_pipeline(use_rust):
    """Run extract_data + full inference once; return (extract_ms, infer_ms)."""
    _ed_mod._HAS_RUST = use_rust
    _ct_mod._HAS_RUST = use_rust
    try:
        t0 = time.perf_counter()
        adata_b = extract_data(
            adata_full,
            BENCH_MODEL,
            dataset_name="bench",
            modality_name_dict={"unspliced": "unspliced", "spliced": "spliced"},
            n_genes=len(BENCH_GENES),
            genes_to_fit=BENCH_GENES,
            hist_type="unique",
            viz=False,
        )
        # Inject gene lengths (pre-loaded metadata, not I/O inside the loop)
        adata_b.var["log_lengths"] = BENCH_LENGTHS
        t_extract = (time.perf_counter() - t0) * 1e3

        t0 = time.perf_counter()
        sd = searchdata_from_adata(adata_b)
        ip = InferenceParameters("bench", BENCH_MODEL, **INFER_PARAMS)
        sr = ip.fit_all_grid_points(sd, num_cores=1, save=False)
        sr.find_sampling_optimum(discard_rejected=False)
        t_infer = (time.perf_counter() - t0) * 1e3
    finally:
        _ed_mod._HAS_RUST = True
        _ct_mod._HAS_RUST = True
    return t_extract, t_infer


print("Running Python pipeline (may take ~60–120 s) ...")
py_ext5, py_inf5 = run_pipeline(use_rust=False)
print(f"  extract_data : {py_ext5:7.0f} ms")
print(f"  inference    : {py_inf5:7.0f} ms")
print(f"  total        : {py_ext5 + py_inf5:7.0f} ms")

print("\nRunning Rust pipeline ...")
ru_ext5, ru_inf5 = run_pipeline(use_rust=True)
print(f"  extract_data : {ru_ext5:7.0f} ms")
print(f"  inference    : {ru_inf5:7.0f} ms")
print(f"  total        : {ru_ext5 + ru_inf5:7.0f} ms")

py_total5 = py_ext5 + py_inf5
ru_total5 = ru_ext5 + ru_inf5
print(f"\nSpeedup — extract_data: {py_ext5/ru_ext5:.1f}x  "
      f"inference: {py_inf5/ru_inf5:.1f}x  "
      f"total: {py_total5/ru_total5:.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels   = ['extract_data', 'inference', 'total']
py_vals  = [py_ext5, py_inf5, py_total5]
ru_vals  = [ru_ext5, ru_inf5, ru_total5]
x        = np.arange(len(labels))
w        = 0.35

ax = axes[0]
ax.bar(x - w/2, py_vals, w, color='steelblue',  label='Python (serial)')
ax.bar(x + w/2, ru_vals, w, color='firebrick',  label='Rust (rayon)')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Wall time (ms)')
ax.set_title(f'{len(BENCH_GENES)} genes  |  {BENCH_MODEL.bio_model}+{BENCH_MODEL.seq_model}  |  3×3 grid')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
speedups5 = [p / r for p, r in zip(py_vals, ru_vals)]
ax.bar(labels, speedups5, color=['steelblue', 'firebrick', 'seagreen'], alpha=0.8)
ax.axhline(1, color='grey', lw=0.8, ls='--')
ax.set_ylabel('Speedup (Python / Rust)')
ax.set_title('Speedup per phase')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1fx'))
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Full inference pipeline: Rust vs Python', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## Benchmark 6 — full inference pipeline

End-to-end timing from raw count matrices to biophysical parameters.

Model: **Bursty + Poisson**  |  5 genes  |  728 cells  |  3×3 = 9 grid points  |  15 iterations / restart

For this small analysis L-BFGS-B Python overhead dominates over raw computation; the
Rust speedup is primarily in `extract_data` (histogram extraction).  For larger analyses
(100+ genes, large grids), both the histogram and model-evaluation speedups compound.

In [ ]:
header = f'{"Scenario":<58} {"Python (ms)":>12} {"Rust (ms)":>10} {"Speedup":>9}'
print(header)
print('-' * len(header))

all_rows = [
    ('Histogram  5k cells × 200 genes, max=20  [counting]',
     py_times[gene_counts.index(200)], ru_times[gene_counts.index(200)]),
    ('Histogram 20k cells × 100 genes, max=20  [counting]',
     py_times2[-1], ru_times2[-1]),
    ('Histogram  real data (gaba)  728 cells × 200 genes',
     py_times4[-1], ru_times4[-1]),
    ('PSS eval   Bursty+Poisson, grid (100×100)',
     py_pss[grid_sizes5.index(100)], ru_pss[grid_sizes5.index(100)]),
    ('PSS eval   Bursty+Poisson, grid (500×500)',
     py_pss[grid_sizes5.index(500)], ru_pss[grid_sizes5.index(500)]),
    ('Pipeline   extract_data  728 cells × 5 genes',
     py_ext5, ru_ext5),
    ('Pipeline   inference     728 cells × 5 genes, 3×3 grid',
     py_inf5, ru_inf5),
    ('Pipeline   total         728 cells × 5 genes, 3×3 grid',
     py_total5, ru_total5),
]
for label, py_t, ru_t in all_rows:
    print(f'{label:<58} {py_t:>12.1f} {ru_t:>10.1f} {py_t/ru_t:>8.1f}x')

## Benchmark 7 — large-scale inference pipeline: 1 000 genes

**Dataset**: `gaba_example.h5ad` — 728 cells × 32 285 genes (mouse GABAergic neurons)  
**Model**: Bursty + Poisson  
**Gene set**: 1 000 randomly selected expressed genes (≥ 10 cells with spliced > 0)  
**Inference**: 3×3 sampling-parameter grid, up to 15 L-BFGS-B iterations, 1 restart

*Purpose*: assess end-to-end scalability of the monod pipeline at a realistic gene count and
identify where the Rust extension helps — and where Python/numba is competitive.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

adata_gaba = ad.read_h5ad("example_h5ad/gaba_example.h5ad")
adata_gaba.var_names_make_unique()          # dataset has duplicate gene names

# Expressed-gene filter: ≥ 10 cells with spliced count > 0
s_gaba = adata_gaba.layers["spliced"]
s_gaba = s_gaba.toarray() if issparse(s_gaba) else s_gaba
expressed7 = np.where((s_gaba > 0).sum(axis=0) >= 10)[0]

rng7 = np.random.default_rng(42)
idx_1000 = rng7.choice(expressed7, size=1000, replace=False)
GENES_1000 = adata_gaba.var_names[idx_1000].tolist()

BENCH_MODEL_7 = CMEModel("Bursty", "Poisson")
INFER_PARAMS_7 = dict(
    use_lengths=False,
    gradient_params={"max_iterations": 15, "init_pattern": "moments", "num_restarts": 1},
    gridsize=[3, 3],
    save=False,
)

print(f"Dataset : {adata_gaba.n_obs} cells × {adata_gaba.n_vars} genes")
print(f"Expressed genes (spliced ≥ 10 cells): {len(expressed7)}")
print(f"Selected for benchmark : {len(GENES_1000)} genes (seed 42)")

In [ ]:
# ── Benchmark 7a: extract_data ───────────────────────────────────────────────
N_REPS7 = 3

def time_extract7(use_rust, n_reps=N_REPS7):
    _ed_mod._HAS_RUST = use_rust
    _ct_mod._HAS_RUST = use_rust
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        ab = extract_data(
            adata_gaba, BENCH_MODEL_7,
            dataset_name="bench7",
            modality_name_dict={"unspliced": "unspliced", "spliced": "spliced"},
            n_genes=len(GENES_1000), genes_to_fit=GENES_1000,
            hist_type="unique", viz=False,
        )
        times.append((time.perf_counter() - t0) * 1e3)
    _ed_mod._HAS_RUST = True
    _ct_mod._HAS_RUST = True
    return float(np.mean(times)), ab

print("Timing extract_data …")
py_ext7, _      = time_extract7(use_rust=False)
ru_ext7, adata7 = time_extract7(use_rust=True)
print(f"  Python : {py_ext7:7.0f} ms   ({py_ext7/len(GENES_1000):.2f} ms / gene)")
print(f"  Rust   : {ru_ext7:7.0f} ms   ({ru_ext7/len(GENES_1000):.2f} ms / gene)")
print(f"  Speedup: {py_ext7/ru_ext7:.2f}×")
print()
print("Note: at 728 cells the histogram step is fast in both backends —")
print("rayon parallelism pays off more for datasets with tens-of-thousands of cells.")

In [ ]:
# ── Grid-size analysis ────────────────────────────────────────────────────────
# PSS grid area = M[unspliced] × M[spliced]; larger = more expensive PSS eval.
sd7 = searchdata_from_adata(adata7)
areas = sd7.M[0] * sd7.M[1]        # shape (n_genes,)

pcts = np.percentile(areas, [10, 25, 50, 75, 90, 99])
print("PSS grid-area distribution (unspliced × spliced) across 1 000 genes:")
print(f"  P10={pcts[0]:.0f}  P25={pcts[1]:.0f}  P50={pcts[2]:.0f}  "
      f"P75={pcts[3]:.0f}  P90={pcts[4]:.0f}  P99={pcts[5]:.0f}  max={areas.max():.0f}")
print()
for thresh in [500, 1000, 5000]:
    n = (areas >= thresh).sum()
    print(f"  Genes with area ≥ {thresh:5d} : {n:4d} / {len(areas)}")
print()
print("Rust PSS outperforms Python/numba for large grids (area ≳ 1 000);")
print("for the majority of this 728-cell dataset the grids are small and")
print("Python/numba's lower call overhead is competitive.")

In [ ]:
# -- Benchmark 7b: inference pipeline -----------------------------------------
def run_inf7(use_rust, genes):
    _ed_mod._HAS_RUST = use_rust
    _ct_mod._HAS_RUST = use_rust
    ab = extract_data(
        adata_gaba, BENCH_MODEL_7,
        dataset_name="bench7",
        modality_name_dict={"unspliced": "unspliced", "spliced": "spliced"},
        n_genes=len(genes), genes_to_fit=genes,
        hist_type="unique", viz=False,
    )
    t0 = time.perf_counter()
    sd = searchdata_from_adata(ab)
    ip = InferenceParameters("bench7", BENCH_MODEL_7, **INFER_PARAMS_7)
    sr = ip.fit_all_grid_points(sd, num_cores=1, save=False)
    elapsed = (time.perf_counter() - t0) * 1e3
    _ed_mod._HAS_RUST = True
    _ct_mod._HAS_RUST = True
    return elapsed, sr

print("Running Python inference on 1 000 genes (Bursty+Poisson, may take ~4 min) ...")
py_inf7_1000, _   = run_inf7(use_rust=False, genes=GENES_1000)
print(f"  Python 1 000 genes : {py_inf7_1000/1e3:6.1f} s  ({py_inf7_1000/1000:.1f} ms / gene)")

print("Running Rust inference on 1 000 genes ...")
ru_inf7_1000, sr7 = run_inf7(use_rust=True, genes=GENES_1000)
print(f"  Rust   1 000 genes : {ru_inf7_1000/1e3:6.1f} s  ({ru_inf7_1000/1000:.1f} ms / gene)")

print()
print(f"Speedup: {py_inf7_1000/ru_inf7_1000:.2f}x")
print()
print("Takeaway: PSS call overhead is significant relative to L-BFGS-B optimizer overhead")
print("for the small grids in this 728-cell dataset.  Rust speedup for inference grows")
print("with dataset size (larger cell counts -> larger grid limits -> bigger PSS workloads).")

In [ ]:
# ── Benchmark 7c: high-expression subset ─────────────────────────────────────
# Select the 50 genes with the largest PSS grid area — these are the genes
# where the Rust PSS speedup should be most visible.
top50_idx = np.argsort(areas)[-50:]
GENES_TOP50 = [sd7.gene_names[i] for i in top50_idx]
print(f"Top-50 high-expression genes  grid area: "
      f"min={areas[top50_idx].min():.0f}  "
      f"median={np.median(areas[top50_idx]):.0f}  "
      f"max={areas[top50_idx].max():.0f}")

py_top50, _ = run_inf7(use_rust=False, genes=GENES_TOP50)
ru_top50, _ = run_inf7(use_rust=True,  genes=GENES_TOP50)
print(f"Python 50 high-expr genes : {py_top50:7.0f} ms  ({py_top50/50:.1f} ms / gene)")
print(f"Rust   50 high-expr genes : {ru_top50:7.0f} ms  ({ru_top50/50:.1f} ms / gene)")
print(f"Speedup (high-expression) : {py_top50/ru_top50:.2f}×")

In [ ]:
# -- Visualise: per-gene timing vs grid area ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: extract_data
ax = axes[0]
labels7e = ["Python\n(serial)", "Rust\n(rayon)"]
vals7e   = [py_ext7, ru_ext7]
bars = ax.bar(labels7e, vals7e, color=["steelblue", "firebrick"], width=0.4)
for bar, v in zip(bars, vals7e):
    ax.text(bar.get_x() + bar.get_width()/2, v + 5, f"{v:.0f} ms",
            ha="center", va="bottom", fontsize=9)
ax.set_title("extract_data -- 1 000 genes, 728 cells", fontsize=11)
ax.set_ylabel("Wall-clock time (ms)")
ax.set_ylim(0, max(vals7e) * 1.4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))

# Right: inference per gene -- all genes vs high-expression
ax = axes[1]
labels7i = [
    "All genes\n(Python, 1000)",
    "All genes\n(Rust, 1000)",
    "High-expr\n(Python, 50)",
    "High-expr\n(Rust, 50)",
]
vals7i = [
    py_inf7_1000 / 1000,
    ru_inf7_1000 / 1000,
    py_top50     / 50,
    ru_top50     / 50,
]
colors7i = ["steelblue", "firebrick", "steelblue", "firebrick"]
bars = ax.bar(labels7i, vals7i, color=colors7i, width=0.5)
for bar, v in zip(bars, vals7i):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.0f}",
            ha="center", va="bottom", fontsize=9)
ax.set_title("Inference time per gene (3x3 grid, 15 iter)", fontsize=11)
ax.set_ylabel("ms / gene")
ax.set_ylim(0, max(vals7i) * 1.4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))

fig.tight_layout()
plt.show()
print()
print(f"Rust speedup -- all genes (1000): {py_inf7_1000/ru_inf7_1000:.2f}x  |  "
      f"high-expression (50): {py_top50/ru_top50:.2f}x")

In [ ]:
# -- Updated summary table ----------------------------------------------------
all_rows_updated = all_rows + [
    ("B7: extract_data  728 cells x 1 000 genes",
     py_ext7, ru_ext7),
    ("B7: inference     1 000 genes (Bursty+Poisson, 3x3)",
     py_inf7_1000, ru_inf7_1000),
    ("B7: inference     high-expr 50 genes",
     py_top50, ru_top50),
]

header2 = f'{"Scenario":<60} {"Python (ms)":>12} {"Rust (ms)":>10} {"Speedup":>9}'
print(header2)
print("-" * len(header2))
for label, py_t, ru_t in all_rows_updated:
    print(f"{label:<60} {py_t:>12.1f} {ru_t:>10.1f} {py_t/ru_t:>8.1f}x")

## Benchmark 8 -- repeat with Bursty + None (no technical noise)

Same 1 000-gene / 728-cell dataset, but fitting the **Bursty** biological model with
`seq_model="None"` -- no Poisson capture-rate model.

Key differences from Benchmark 7:
* **No sampling grid**: only one grid point (`gridsize=[1, 1]`), so inference has
  roughly 9x fewer PSS evaluations per gene than the 3x3 Poisson grid.
* **Rust fast-path** uses the `seq_model="None"` branch (`_fast_cond`), which avoids
  the Poisson-mesh transformation and applies rayon parallelism directly on the
  base mesh.
* **Context**: Bursty+None is appropriate when capture-rate variation is negligible
  or has been pre-corrected (e.g., depth-normalised data).

In [ ]:
# -- Setup --------------------------------------------------------------------
# Reuse GENES_1000 and adata_gaba from Benchmark 7.
BENCH_MODEL_8  = CMEModel("Bursty", "None")
INFER_PARAMS_8 = dict(
    use_lengths=False,
    gradient_params={"max_iterations": 15, "init_pattern": "moments", "num_restarts": 1},
    gridsize=[1, 1],   # no sampling parameter grid for seq_model="None"
    save=False,
)
print(f"Model: {BENCH_MODEL_8.bio_model} + {BENCH_MODEL_8.seq_model}")
print(f"Inference grid: {INFER_PARAMS_8['gridsize']}  "
      f"(1 point; Poisson grid is [3, 3] = 9 points in Benchmark 7)")

In [ ]:
# -- Benchmark 8a: extract_data -----------------------------------------------
def time_extract8(use_rust, n_reps=3):
    _ed_mod._HAS_RUST = use_rust
    _ct_mod._HAS_RUST = use_rust
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        ab = extract_data(
            adata_gaba, BENCH_MODEL_8,
            dataset_name="bench8",
            modality_name_dict={"unspliced": "unspliced", "spliced": "spliced"},
            n_genes=len(GENES_1000), genes_to_fit=GENES_1000,
            hist_type="unique", viz=False,
        )
        times.append((time.perf_counter() - t0) * 1e3)
    _ed_mod._HAS_RUST = True
    _ct_mod._HAS_RUST = True
    return float(np.mean(times)), ab

print("Timing extract_data (Bursty+None) ...")
py_ext8, _      = time_extract8(use_rust=False)
ru_ext8, adata8 = time_extract8(use_rust=True)
print(f"  Python : {py_ext8:7.0f} ms")
print(f"  Rust   : {ru_ext8:7.0f} ms")
print(f"  Speedup: {py_ext8/ru_ext8:.2f}x  (histogram extraction unchanged)")

In [ ]:
# -- Benchmark 8b: direct PSS evaluation -- crossover grid size ---------------
# For seq_model="None" the Rust fast-path bypasses Poisson mesh transform.
# We sweep grid sizes to find where Rust overtakes Python/numba.
p_bio8 = np.array([0.0, 0.0, 0.0])       # b=1, beta=1, gamma=1
SIZES8 = [20, 30, 50, 80, 100, 200, 400]

py_pss8, ru_pss8 = [], []
for sz in SIZES8:
    lims = np.array([sz, sz])
    for mode, flag, store in [("Python", False, py_pss8), ("Rust", True, ru_pss8)]:
        _ct_mod._HAS_RUST = flag
        # warm up
        for _ in range(2):
            BENCH_MODEL_8.eval_model_pss(p_bio8, lims, samp=None)
        ts = []
        for _ in range(5):
            t0 = time.perf_counter()
            BENCH_MODEL_8.eval_model_pss(p_bio8, lims, samp=None)
            ts.append((time.perf_counter() - t0) * 1e3)
        store.append(float(np.mean(ts)))
_ct_mod._HAS_RUST = True

print(f"{'Grid':>8}  {'Python (ms)':>12}  {'Rust (ms)':>10}  {'Speedup':>9}")
print("-" * 46)
for sz, py_t, ru_t in zip(SIZES8, py_pss8, ru_pss8):
    print(f"{sz:>3}x{sz:<3}  {py_t:>12.2f}  {ru_t:>10.2f}  {py_t/ru_t:>8.2f}x")

In [ ]:
# -- Benchmark 8c: inference pipeline -----------------------------------------
def run_inf8(use_rust, genes):
    _ed_mod._HAS_RUST = use_rust
    _ct_mod._HAS_RUST = use_rust
    ab = extract_data(
        adata_gaba, BENCH_MODEL_8,
        dataset_name="bench8",
        modality_name_dict={"unspliced": "unspliced", "spliced": "spliced"},
        n_genes=len(genes), genes_to_fit=genes,
        hist_type="unique", viz=False,
    )
    t0 = time.perf_counter()
    sd = searchdata_from_adata(ab)
    ip = InferenceParameters("bench8", BENCH_MODEL_8, **INFER_PARAMS_8)
    sr = ip.fit_all_grid_points(sd, num_cores=1, save=False)
    elapsed = (time.perf_counter() - t0) * 1e3
    _ed_mod._HAS_RUST = True
    _ct_mod._HAS_RUST = True
    return elapsed, sr, sd

print("Python inference -- 1 000 genes, Bursty+None ...")
py_inf8_1000, _, sd8 = run_inf8(use_rust=False, genes=GENES_1000)
print(f"  {py_inf8_1000/1e3:6.1f} s  ({py_inf8_1000/1000:.1f} ms / gene)")

print("Rust inference -- 1 000 genes, Bursty+None ...")
ru_inf8_1000, _, _   = run_inf8(use_rust=True, genes=GENES_1000)
print(f"  {ru_inf8_1000/1e3:6.1f} s  ({ru_inf8_1000/1000:.1f} ms / gene)")

print(f"\nSpeedup (1 000 genes): {py_inf8_1000/ru_inf8_1000:.2f}x")

# High-expression genes
areas8    = sd8.M[0] * sd8.M[1]
top50_8   = np.argsort(areas8)[-50:]
GENES_TOP50_8 = [sd8.gene_names[i] for i in top50_8]
print(f"\nTop-50 high-expression genes: "
      f"area min={areas8[top50_8].min()} median={np.median(areas8[top50_8]):.0f} max={areas8[top50_8].max()}")

print("Python inference -- top-50 high-expr ...")
py_top8, _, _ = run_inf8(use_rust=False, genes=GENES_TOP50_8)
print(f"  {py_top8:7.0f} ms  ({py_top8/50:.1f} ms / gene)")

print("Rust inference -- top-50 high-expr ...")
ru_top8, _, _ = run_inf8(use_rust=True, genes=GENES_TOP50_8)
print(f"  {ru_top8:7.0f} ms  ({ru_top8/50:.1f} ms / gene)")

print(f"\nSpeedup -- all genes (1000): {py_inf8_1000/ru_inf8_1000:.2f}x  |  "
      f"high-expression (50): {py_top8/ru_top8:.2f}x")
print()
print("Note: with gridsize=[1,1] the per-gene PSS call count is ~9x lower than")
print("Benchmark 7, so scipy optimizer overhead is proportionally larger. The")
print("Rust speedup for direct PSS eval scales strongly with grid size (see 8b).")

In [ ]:
# -- Visualise: PSS crossover + inference comparison --------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: direct PSS speedup vs grid area for both models
ax = axes[0]
speedup8 = [py/ru for py, ru in zip(py_pss8, ru_pss8)]
ax.semilogx([s**2 for s in SIZES8], speedup8, 's-', color='firebrick', label='Bursty+None (B8)')
try:
    speedup5 = [py/ru for py, ru in zip(py_pss, ru_pss)]
    ax.semilogx([s**2 for s in grid_sizes5], speedup5, 'o--', color='steelblue',
                label='Bursty+Poisson (B5)')
except NameError:
    pass
ax.axhline(1.0, color='grey', linestyle=':', linewidth=1, label='Breakeven')
ax.set_xlabel("PSS grid area (M0 x M1)")
ax.set_ylabel("Speedup (Python / Rust)")
ax.set_title("PSS eval speedup vs grid size", fontsize=11)
ax.legend(fontsize=9)

# Right: per-gene inference time -- Poisson vs None
ax = axes[1]
categories = ["B7 Poisson\nall (1000)", "B7 Poisson\nhigh (50)",
              "B8 None\nall (1000)", "B8 None\nhigh (50)"]
py_v = [py_inf7_1000/1000, py_top50/50, py_inf8_1000/1000, py_top8/50]
ru_v = [ru_inf7_1000/1000, ru_top50/50, ru_inf8_1000/1000, ru_top8/50]
x_pos = np.arange(len(categories))
w = 0.35
ax.bar(x_pos - w/2, py_v, w, label='Python', color='steelblue')
ax.bar(x_pos + w/2, ru_v, w, label='Rust',   color='firebrick')
for xi, (pv, rv) in enumerate(zip(py_v, ru_v)):
    ax.text(xi - w/2, pv + 1, f"{pv:.0f}", ha='center', va='bottom', fontsize=8)
    ax.text(xi + w/2, rv + 1, f"{rv:.0f}", ha='center', va='bottom', fontsize=8)
ax.set_xticks(x_pos)
ax.set_xticklabels(categories, fontsize=8)
ax.set_ylabel("ms / gene")
ax.set_title("Inference per-gene: Bursty+Poisson vs Bursty+None", fontsize=11)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()